In [1]:
# Extract images for all items in the items in the dataset

import json
metadata_dict = {}
file ="data/meta_All_Beauty.jsonl" # e.g., "All_Beauty.jsonl", downloaded from the `review` link above
with open(file, 'r') as fp:
    for line in fp:
        # try:
        tmp_dict = {'images':[]}
        data = json.loads(line.strip())
        ctr =0
        for img_dict in data['images']:
            # if type(tmp_dict['images']) == list:
            tmp_dict['images'].append(img_dict.get("large", img_dict.get("hi-res", "")))
            
        # except:
        #     print(line.strip())
        metadata_dict[data['parent_asin']] = tmp_dict

In [14]:
metadata_dict['0701169850']

{'images': ['https://m.media-amazon.com/images/I/41X61VPJYKL._SX334_BO1,204,203,200_.jpg']}

In [ ]:
import requests, os
from concurrent.futures import ThreadPoolExecutor

folder_location = "data/All_Beauty/"

os.makedirs(folder_location, exist_ok=True)

def download_image(product_id, idx, image_url, folder_location):
    try:
        # Fetch the image
        response = requests.get(image_url, stream=True)
        response.raise_for_status()  # Raise an error for bad status codes

        # Create a unique filename
        filename = f"{product_id}_{idx + 1}.jpg"
        file_path = os.path.join(folder_location, filename)

        # Save the image
        with open(file_path, 'wb') as file:
            for chunk in response.iter_content(1024):
                file.write(chunk)
        print(f"Downloaded: {file_path}")
    except Exception as e:
        print(f"Failed to download {image_url}: {e}")

# Use ThreadPoolExecutor for parallel downloads
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = []
    folder_content = os.listdir(folder_location)
    # Check if images already exist
    existing_images = [f.split("_")[0] for f in os.listdir(folder_location) if f.endswith('.jpg')]
    existing_images = list(set(existing_images))
    
    
    with open("data/s.csv", "r") as f:
        file_contents = f.read()
    contents_list = file_contents.split(",")
    
    
    for product_id in contents_list:
        # Download up to 2 images
        for idx, image_url in enumerate(metadata_dict[product_id]['images'][:2]):
            futures.append(executor.submit(download_image, product_id, idx, image_url, folder_location))

## Amazon Books

In [1]:
# Extract images for all items in the items in the dataset

import json
import gzip

metadata_dict = {}

filename = 'data/meta_Books.jsonl.gz'  # Assuming the file is locally accessible
json_content = []
with gzip.open(filename, 'rb') as gzip_file:  # Open in binary read mode
    for line in gzip_file:
        tmp_dict = {'images':[], 'title':None, 'description':None}
        line = line.strip()  # Remove leading/trailing whitespace
        if line:
            data = json.loads(line)  # Parse each line as JSON
            tmp_dict['title'] =data['title'] 
            tmp_dict['description'] =data['description']
            # json_content.append(obj)
            for img_dict in data['images']:
            # if type(tmp_dict['images']) == list:
                tmp_dict['images'].append(img_dict.get("large", img_dict.get("hi-res", "")))
                 
            
        # except:
        #     print(line.strip())
        metadata_dict[data['parent_asin']] = tmp_dict

        

In [8]:
with open("data/metadata_Books.json", "w") as f:
    json.dump(metadata_dict, f)

### Get the images

In [6]:
import json
# metadata_dict = json.load()
with open("data/metadata_Books.json", 'r') as file:
    metadata_dict = json.load(file)

In [ ]:
import requests, os
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from tqdm import tqdm


folder_location = "data/All_Books/"
csv_file = 'data/Books.train.csv.gz'

os.makedirs(folder_location, exist_ok=True)

def download_image(product_id, idx, image_url, folder_location):
    try:
        # Fetch the image
        response = requests.get(image_url, stream=True)
        response.raise_for_status()  # Raise an error for bad status codes

        # Create a unique filename
        filename = f"{product_id}_{idx + 1}.jpg"
        file_path = os.path.join(folder_location, filename)

        # Save the image
        with open(file_path, 'wb') as file:
            for chunk in response.iter_content(1024):
                file.write(chunk)
        # print(f"Downloaded: {file_path}")
    except Exception as e:
        print(f"Failed to download {image_url}: {e}")

# Use ThreadPoolExecutor for parallel downloads
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = []
    folder_content = os.listdir(folder_location)
    # Check if images already exist
    existing_images = [f.split("_")[0] for f in os.listdir(folder_location) if f.endswith('.jpg')]
    existing_images = list(set(existing_images))
    
    
    df = pd.read_csv(csv_file)
    contents_list = df['parent_asin']
    
    
    # for product_id in contents_list:
    # Wrap contents_list with tqdm for a progress bar
    for product_id in tqdm(contents_list, desc="Downloading images"):
        # Download up to 2 images
        for idx, image_url in enumerate(metadata_dict[product_id]['images'][:2]):
            futures.append(executor.submit(download_image, product_id, idx, image_url, folder_location))

Failed to download https://m.media-amazon.com/images/I/51jEmzNkNVL._SX398_BO1,204,203,200_.jpg: 404 Client Error: Not Found for url: https://m.media-amazon.com/images/I/51jEmzNkNVL._SX398_BO1,204,203,200_.jpg
Failed to download https://m.media-amazon.com/images/I/51nm8pUEAML._SX383_BO1,204,203,200_.jpg: 404 Client Error: Not Found for url: https://m.media-amazon.com/images/I/51nm8pUEAML._SX383_BO1,204,203,200_.jpg


In [9]:

df = pd.read_csv('data/Books.train.csv.gz')

In [10]:
df.head()

,user_id,parent_asin,rating,timestamp,history
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1446304000,5.0,1441260345000,NaN
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1564770672,5.0,1441260365000,1446304000
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1442450703,5.0,1523093714024,1446304000 1564770672
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1780671067,1.0,1611623223325,1446304000 1564770672 1442450703
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1645671127,3.0,1612044209266,1446304000 1564770672 1442450703 1780671067
